# Colombia Open — Directorio de establecimientos educativos

Este notebook muestra cómo cargar el dataset **`educacion_establecimientos`** (directorio único de establecimientos educativos 2015-2025, MEN) con `colombia-open`: por establecimiento, año y municipio.

> **Nota de volumen:** este dataset es mucho más pesado que los demás del catálogo (~606,206 filas). El bundle publicado ya lo trae pre-construido, así que `pip install colombia-open` + la primera carga solo paga el costo de descargar y cachear el bundle una vez (no el fetch paginado en vivo de ~7 minutos que sí corre `make build` localmente) — ver `src/extractors/educacion_establecimientos_extractor.py` si corrés el pipeline vos mismo.

> **Nota (2026-10):** `colombia-open` ya está publicado en [PyPI](https://pypi.org/project/colombia-open/) — `pip install colombia-open` descarga y cachea el bundle de datos publicado en GitHub Releases automáticamente (ver `README.md` del repo, sección "Estado del proyecto").

## Setup

In [ ]:
!pip install -q colombia-open

## Cargar el dataset

In [ ]:
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
educacion = hub.load_polars("educacion_establecimientos")

print(educacion.shape)
educacion.head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `anio` | int | Año del reporte, 2015 a 2025. |
| `cod_dpto` / `cod_mpio` | string (2 / 5 dígitos) | Códigos DIVIPOLA — join key hacia `municipios`. |
| `codigo_dane_establecimiento` | string (12 dígitos) | Identificador del establecimiento. **No confiable como clave única en 2023** — ver limitaciones abajo. |
| `nombre_establecimiento` | string | Nombre del establecimiento. |
| `sector` | string | `"OFICIAL"` / `"NO_OFICIAL"`. |
| `total_matricula` | int, nullable | Matrícula total reportada ese año. |
| `cantidad_sedes` | int, nullable | Número de sedes del establecimiento. |

Esquema completo: [`docs/datasets/educacion_establecimientos.md`](../../docs/datasets/educacion_establecimientos.md).

## Establecimientos oficiales de un municipio en un año

In [ ]:
# Establecimientos oficiales de Medellín (cod_mpio == "05001") en 2024
educacion.filter(
    (educacion["cod_mpio"] == "05001")
    & (educacion["anio"] == 2024)
    & (educacion["sector"] == "OFICIAL")
).select(["nombre_establecimiento", "total_matricula", "cantidad_sedes"]).sort(
    "total_matricula", descending=True
).head(10)

## Join con `municipios`

In [ ]:
joined = hub.cross_view(["educacion_establecimientos", "municipios"])
joined.filter((joined["anio"] == 2024) & (joined["cod_mpio"] == "76001")).select(
    ["nombre_establecimiento", "nombre_mpio", "sector", "total_matricula"]
).head(5)

## Validación y el defecto conocido de 2023

`validate_dataset()` detecta dinámicamente, año por año, si `codigo_dane_establecimiento` tiene cardinalidad anormalmente baja (el defecto real de la fuente para 2023 — miles de establecimientos distintos comparten el mismo código) y lo reporta como **warning**, no como error de clave duplicada.

In [ ]:
result = hub.validate_dataset("educacion_establecimientos")
print("status:", result["status"])
for warning in result["warnings"]:
    print("warning:", warning)

## Limitaciones conocidas

- `codigo_dane_establecimiento` no es una clave única confiable para el año 2023 (defecto de la fuente, no de este pipeline).
- 25 filas (todas de 2023) no traen `cod_mpio` — el campo viene ausente en la fuente, no se fabrica un valor de reemplazo.
- `cod_mpio` puede referenciar un municipio que no existe en el `municipios` vigente (DIVIPOLA) — son dos productos de entidades distintas (MEN vs. DANE) con ciclos de actualización propios.

Documentación completa: [`docs/datasets/educacion_establecimientos.md`](../../docs/datasets/educacion_establecimientos.md).